# Fit a real normal-source dynamics model

This notebook performs a small real ridge fit and then inspects the complete saved nested source evaluation. It does not retune the published experiment. Inner folds select settings and outer folds provide held-out source predictions.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/restart_data.py').exists()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/restart_data.py').exists())
sys.path.insert(0, str(PROJECT))
from worldmodels.settings import configure
PROJECT, REPOSITORY = configure()
DATA = PROJECT/'artifacts/runs/restart-full10hz-v1'
RUN = PROJECT/'artifacts/runs/restart-dynamics-v1'
assert (DATA/'preparation.json').exists(), 'Run the full extraction command in README first.'
meta = pd.read_csv(DATA/'windows.csv')
with np.load(DATA/'windows.npz', allow_pickle=False) as z:
    x, valid = z['x'], z['valid']
print('Real GAVD observations:', x.shape, 'sources:', meta.source_id.nunique())


Use only normal source observations for this didactic model fit. Its training error is not held-out evidence.

In [ ]:
from worldmodels.restart_experiment import features,targets,fit_grid,hierarchical_weights,predict
training=np.flatnonzero(meta.label.eq('normal'))[:120]
f=features(x[training],valid[training],0,'bilateral')[0]
y,m=targets(x[training],valid[training],0)
models=fit_grid(f,y,m,hierarchical_weights(meta.iloc[training]),alphas=[10.])
pred=predict(models[10.],f)
print('Real fitted coefficient predictions:',pred.shape,'observed training residual MSE:',float(((pred-y)[m]**2).mean()))

Every source remains in one outer fold. The complete run saves normal-only training source lists for every model.

In [ ]:
cohort=pd.read_csv(RUN/'cohort.csv')
assert cohort.groupby('source_id').outer_fold.nunique().max()==1
print('Prepared windows:',len(meta),'model windows:',len(cohort),'support exclusions:',len(meta)-len(cohort))
display(pd.read_csv(RUN/'support_exclusions.csv'))
display(cohort.groupby(['outer_fold','label']).source_id.nunique().unstack(fill_value=0))

The prospective shrinkage rule may select persistence. That choice is reported instead of being hidden.

In [ ]:
selection=json.loads((RUN/'selection.json').read_text())
display(pd.DataFrame(selection).drop(columns=['all_grid_mse'],errors='ignore'))

The fitted run pins input data, protocol and implementation. These identities allow an independent replay to detect stale artifacts.

In [ ]:
display(json.loads((RUN/'identity.json').read_text()))